# Modeling — Retail Demand Forecasting

*Version consolidée (M7) : utilise les modules `src/` et la config centralisée `src/config.py`.*

*L'historique exploratoire complet (M5/M6) n'est plus disponible séparément — ce notebook reflète le pipeline final validé.*

## 1. Setup

In [1]:
import sys
import os
import gc
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from dotenv import load_dotenv

project_root = r"c:\Users\admin\demand-forecasting"
load_dotenv(os.path.join(project_root, ".env"))
sys.path.append(project_root)


In [2]:
from src import config
from src.data.loaders import load_parquet, load_many
from src.features.engineering import add_lag_features, add_rolling_mean
from src.models.train import prepare_features, train_lgbm
from src.models.evaluate import wape, mase, compute_bias


## 2. Chargement des données

*`full` contient déjà toutes les features finalisées (lags, rolling mean, prix, catégorie, calendaire).*

In [3]:
full = load_parquet(config.PROJECT_ROOT, "full")
print(full.shape)
print(full.columns.tolist())


(58327370, 13)
['item_id', 'store_id', 'quantite', 'la_date', 'lag_1', 'lag_7', 'rolling_mean_7', 'price', 'categorie', 'prix_connu', 'is_weekend', 'is_holiday', 'jour_semaine_num']


## 3. Split train / test

*Cutoff défini dans `config.CUTOFF_DATE`, cohérent avec M4-M6.*

In [4]:
print(full.memory_usage(deep=True).sum() / 1e9, "Go")

# Réduction des types numériques
for col in full.select_dtypes(include='int64').columns:
    full[col] = pd.to_numeric(full[col], downcast='integer')
for col in full.select_dtypes(include='float64').columns:
    full[col] = pd.to_numeric(full[col], downcast='float')

print(full.memory_usage(deep=True).sum() / 1e9, "Go")

5.409524142 Go
2.959774602 Go


In [5]:
mask_train = full['la_date'] < config.CUTOFF_DATE
train_fe = full.loc[mask_train].dropna(subset=['lag_1', 'lag_7'])
test_fe = full.loc[~mask_train]

## 4. Préparation des features pour LightGBM

*Colonnes définies dans `config.FEATURES` / `config.CAT_FEATURES`.*

In [6]:
X_train = prepare_features(train_fe, config.FEATURES, config.CAT_FEATURES)
y_train = train_fe[config.TARGET]

X_test = prepare_features(test_fe, config.FEATURES, config.CAT_FEATURES)
y_test = test_fe[config.TARGET]

print(X_train.dtypes)


lag_1                float32
lag_7                float32
rolling_mean_7       float32
price                float32
prix_connu              int8
categorie           category
jour_semaine_num    category
is_weekend          category
is_holiday          category
dtype: object


## 5. Entraînement du modèle final

*Hyperparamètres définis dans `config.MODEL_PARAMS` (alpha=0.63, retenu en M6).*

In [7]:
model = train_lgbm(X_train, y_train, config.CAT_FEATURES, **config.MODEL_PARAMS)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.013869 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 706
[LightGBM] [Info] Number of data points in the train set: 56558950, number of used features: 9


## 6. Évaluation — WAPE global et biais

In [8]:
y_pred = model.predict(X_test).clip(min=0)

wape_score = wape(y_test, pd.Series(y_pred, index=y_test.index))
biais_global = compute_bias(y_test, pd.Series(y_pred, index=y_test.index))

print(f"WAPE : {wape_score:.4f} (seuil cible : {config.WAPE_THRESHOLD})")
print(f"Biais global : {biais_global:.4f}")

if wape_score < config.WAPE_THRESHOLD:
    print("OK — sous le seuil 'prêt à déployer'")
else:
    print("ATTENTION — au-dessus du seuil 'prêt à déployer'")


WAPE : 0.6890 (seuil cible : 0.7)
Biais global : -0.0397
OK — sous le seuil 'prêt à déployer'


## 7. Évaluation — WAPE par catégorie

In [9]:
test_fe['y_pred'] = y_pred

wape_par_categorie = test_fe.groupby('categorie').apply(
    lambda g: wape(g['quantite'], g['y_pred'])
)
print(wape_par_categorie)

depassements = wape_par_categorie[wape_par_categorie > config.WAPE_PER_CATEGORY_MAX]
if len(depassements) > 0:
    print(f"ATTENTION — catégories au-dessus de {config.WAPE_PER_CATEGORY_MAX} :")
    print(depassements)


categorie
FOODS        0.631483
HOBBIES      0.962907
HOUSEHOLD    0.746502
dtype: float64


## 8. Évaluation — MASE par série et par quartile de volume

*Le WAPE agrégé favorise les séries à fort volume. Le MASE, calculé par série, révèle la performance réelle sur la longue traîne.*

In [10]:
naive_scale = (
    train_fe.groupby(['item_id', 'store_id'])
    .apply(lambda g: (g['quantite'] - g['lag_1']).abs().mean())
    .rename('naive_scale')
)

mae_model = (
    test_fe.groupby(['item_id', 'store_id'])
    .apply(lambda g: (g['quantite'] - g['y_pred']).abs().mean())
    .rename('mae_model')
)

df_mase = pd.concat([mae_model, naive_scale], axis=1)
df_mase['mase'] = df_mase.apply(lambda r: mase(r['mae_model'], r['naive_scale']), axis=1)

df_mase['volume_quartile'] = pd.qcut(df_mase['naive_scale'], 4, labels=['Q1 (faible)', 'Q2', 'Q3', 'Q4 (fort)'])

print(df_mase.groupby('volume_quartile')['mase'].median())
print(df_mase.groupby('volume_quartile').apply(lambda g: (g['mase'] < 1).mean()))


volume_quartile
Q1 (faible)    1.475114
Q2             1.228075
Q3             1.031333
Q4 (fort)      0.864229
Name: mase, dtype: float64
volume_quartile
Q1 (faible)    0.366352
Q2             0.399002
Q3             0.472077
Q4 (fort)      0.650210
dtype: float64


## 9. Évaluation — biais par quartile de volume (segment critique : Q4)

In [11]:
biais_par_serie = test_fe.groupby(['item_id', 'store_id']).apply(
    lambda g: compute_bias(g['quantite'], g['y_pred'])
).rename('biais')

df_biais = df_mase[['volume_quartile']].join(biais_par_serie)
print(df_biais.groupby('volume_quartile')['biais'].mean())


volume_quartile
Q1 (faible)   -0.085991
Q2            -0.080719
Q3            -0.025831
Q4 (fort)      0.033896
Name: biais, dtype: float64


## 10. Backtesting multi-fenêtres

*Fenêtres définies dans `config.BACKTEST_WINDOWS`. Critère du cadrage M0 : dégradation < `config.BACKTEST_MAX_DEGRADATION` entre fenêtres.*

In [12]:
results_backtest = []

for train_end, test_end in config.BACKTEST_WINDOWS:
    train_bt = full[full['la_date'] < train_end].dropna(subset=['lag_1', 'lag_7'])
    test_bt = full[(full['la_date'] >= train_end) & (full['la_date'] < test_end)]

    X_train_bt = prepare_features(train_bt, config.FEATURES, config.CAT_FEATURES)
    y_train_bt = train_bt[config.TARGET]
    X_test_bt = prepare_features(test_bt, config.FEATURES, config.CAT_FEATURES)
    y_test_bt = test_bt[config.TARGET]

    model_bt = train_lgbm(X_train_bt, y_train_bt, config.CAT_FEATURES, **config.MODEL_PARAMS)

    y_pred_bt = model_bt.predict(X_test_bt).clip(min=0)
    wape_bt = wape(y_test_bt, pd.Series(y_pred_bt, index=y_test_bt.index))
    biais_bt = compute_bias(y_test_bt, pd.Series(y_pred_bt, index=y_test_bt.index))

    results_backtest.append({
        'train_end': train_end, 'test_end': test_end,
        'wape': wape_bt, 'biais_global': biais_bt,
        'n_train': len(train_bt), 'n_test': len(test_bt)
    })
    print(f"Train < {train_end} | Test [{train_end}, {test_end}) | WAPE={wape_bt:.4f} | biais={biais_bt:.4f}")

df_backtest = pd.DataFrame(results_backtest)

degradation = (df_backtest['wape'].max() - df_backtest['wape'].min()) / df_backtest['wape'].min()
print(f"\nDégradation max entre fenêtres : {degradation:.1%} (seuil : {config.BACKTEST_MAX_DEGRADATION:.0%})")
print(df_backtest)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 3.133129 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 707
[LightGBM] [Info] Number of data points in the train set: 55034450, number of used features: 9
Train < 2016-01-15 | Test [2016-01-15, 2016-03-05) | WAPE=0.6913 | biais=-0.0483
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.272524 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 708
[LightGBM] [Info] Number of data points in the train set: 55552780, number of used features: 9
Train < 2016-02-01 | Test [2016-02-01, 2016-03-22) | WAPE=0.6909 | biais=-0.0390
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.186788 seconds.
You can set `force_row_wis

## 11. Sauvegarde du modèle final

In [13]:
os.makedirs(config.MODEL_DIR, exist_ok=True)
joblib.dump(model, os.path.join(config.MODEL_DIR, config.MODEL_FILENAME))
print(f"Modèle sauvegardé : {os.path.join(config.MODEL_DIR, config.MODEL_FILENAME)}")


Modèle sauvegardé : c:\Users\admin\demand-forecasting\models\lgbm_m6_quantile_alpha063.pkl
